In [1]:
import kagglehub
import pandas as pd
import os
from collections import Counter
from langdetect import detect
import random

c:\Users\aryan\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Download the dataset
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
print("Path to dataset files:", path)

# Load the dataset into a dataframe
data = pd.read_csv(path + "/twcs/twcs.csv")

Path to dataset files: C:\Users\aryan\.cache\kagglehub\datasets\thoughtvector\customer-support-on-twitter\versions\10


In [3]:
data.head(20)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0
5,6,sprintcare,False,Tue Oct 31 21:46:24 +0000 2017,@115712 Can you please send us a private messa...,"5,7",8.0
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN
7,11,sprintcare,False,Tue Oct 31 22:10:35 +0000 2017,@115713 This is saddening to hear. Please shoo...,NaN,12.0
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...,"11,13,14",15.0
9,15,sprintcare,False,Tue Oct 31 20:03:31 +0000 2017,@115713 We understand your concerns and we'd l...,12,16.0


From the data I have understood that

1. Inbound refers to tweets that was recieved by companies, hence True and outbond are those tweets which are tweeted by companies in response to users, which is True.

2. If in_response_to_tweet_id is Nan, that means those were the root threads.

3. If response_tweet_id is Nan, that means those were the last threads in the conversation.

4. Inorder to make underatsnding more eay, we need to map all those threads together.

In [4]:
print("Shape of the data is: ", data.shape)
print("Columns in the data are: ", data.columns.tolist())
print(data.info())

Shape of the data is:  (2811774, 7)
Columns in the data are:  ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB
None


Convert in_response_tweet_id to Int64, created_at to date time format and response_tweet_id into a list

In [5]:
data["created_at"] = pd.to_datetime(data['created_at'])
data["in_response_to_tweet_id"] = data["in_response_to_tweet_id"].astype("Int64")

def parseResponseTweetIds(val):
    if pd.isna(val):
        return []
    return [int(x) for x in str(val).split(",")]

data["response_tweet_id"] = data["response_tweet_id"].apply(parseResponseTweetIds)

C:\Users\aryan\AppData\Local\Temp\ipykernel_5672\1826347718.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["created_at"] = pd.to_datetime(data['created_at'])


In [6]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype              
---  ------                   -----              
 0   tweet_id                 int64              
 1   author_id                object             
 2   inbound                  bool               
 3   created_at               datetime64[ns, UTC]
 4   text                     object             
 5   response_tweet_id        object             
 6   in_response_to_tweet_id  Int64              
dtypes: Int64(1), bool(1), datetime64[ns, UTC](1), int64(1), object(3)
memory usage: 134.1+ MB
None


In [7]:
# Find the number of times author_id appears in the data
print("Number of times author_id appears in the data is: ", data['author_id'].value_counts())

Number of times author_id appears in the data is:  author_id
AmazonHelp      169840
AppleSupport    106860
Uber_Support     56270
SpotifyCares     43265
Delta            42253
                 ...  
456282               1
456281               1
456280               1
456276               1
823870               1
Name: count, Length: 702777, dtype: int64


As AmazonHelp has the highest volume of tweets, I decided with go with the brand "AmazonHelp"

Also these are just the tweets that have been sent by AmazonHelp, we need to get all the tweets in the thread for a better context.

In [8]:
tweet_index = data.set_index('tweet_id')

roots = data[data['in_response_to_tweet_id'].isna()]['tweet_id'].tolist()

def collectThreads(rootId):
    threadTweets = []
    frontier = [rootId]
    seen = set()
    while frontier:
        nextFrontier = []

        for tid in frontier:
            if tid in seen or tid not in tweet_index.index:
                continue

            seen.add(tid)

            row = tweet_index.loc[tid]
            threadTweets.append({
                "tweet_id": tid,
                "author_id": row['author_id'],
                "inbound": row['inbound'],
                "text": row["text"],
                "created_at": row["created_at"],
            })
            children = row["response_tweet_id"]
            if children:
                nextFrontier.extend(children)
        frontier = nextFrontier
    threadTweets.sort(key=lambda x: x["created_at"])
    return threadTweets

amazonThreads = []
for rootId in roots:
    tweets = collectThreads(rootId)
    if any(t['author_id'] == 'AmazonHelp' for t in tweets):
        amazonThreads.append({"thread_id": str(rootId), "tweets": tweets})

print(f"{len(amazonThreads)} Amazon threads found in sample of {len(roots)} roots")


81902 Amazon threads found in sample of 794335 roots


In [9]:
# Checking the languages used in the threads
def detectLanguage(text):
    try:
        return detect(text)
    except:
        return "unknown"

langs = [detectLanguage(t["tweets"][0]["text"]) for t in amazonThreads]
langCounts = Counter(langs)

total = len(langs)

for lang, count in langCounts.most_common(15):
    print(f"{lang}: {count} ({count/total*100:.1f}%)")

en: 61275 (74.8%)
ja: 6511 (7.9%)
es: 4144 (5.1%)
fr: 3292 (4.0%)
de: 2516 (3.1%)
pt: 1617 (2.0%)
it: 920 (1.1%)
nl: 325 (0.4%)
hu: 269 (0.3%)
unknown: 124 (0.2%)
da: 111 (0.1%)
so: 71 (0.1%)
tl: 70 (0.1%)
ca: 69 (0.1%)
af: 63 (0.1%)


So we can see that around 75% of the threads are in English and the rest 25% are a long tail of different languages. I would scope this project to English only and just train the agent on English language and make it a single language agent rather than multi language agent.

In [10]:
# Considering only English threads for further analysis
amazonThreads = [t for t in amazonThreads if detectLanguage(t["tweets"][0]["text"]) == "en"]

In [11]:
import random

def getOpeningText(thread):
    first = thread["tweets"][0]
    if not first["inbound"]:
        return None
    return first["text"]
    
random.seed(42)

sampleForReading = random.sample(amazonThreads, 150)

for t in sampleForReading:
    firstText = getOpeningText(t)
    print(f"[{t['thread_id']}] {firstText}")
    print("---")

[2094035] How the hell do you contact @115821 #CustomerService.... just want my parcel 👎 #whereismydelivery #nothappy
---
[319499] @115850 @115821  Order403-2429419-5499503 Product misold.Amazon not ready to exchange product for another.Highly disappointed wid service.
---
[98579] Buy @115830 prime services for the quick delivery times. Then get a notification that the delivery won’t turn up for another week. Uhhhhhhhm what happened to prime next day 🤔
---
[2428161] @115850 @AmazonHelp Yeah I know that I missed introdctory offer.  Just want to check is there any possibility of getting same price now. https://t.co/MIH0kKKQgo
---
[739083] It's really sad that I lost the bet of rs 500 with my sis cos @115850 bad delivery system I bet that today product will deliver but nt yet https://t.co/BtGoEEEiil
---
[641658] @115821 I used to be your #1 fan. Too many broken promises of 2-day shipping. I'm tired of the disappointment and date of delivery changes!
---
[570014] Dear @AmazonHelp i purchas